# Future Sales Prediction

**Tabular Regression Project** — Predict future sales from historical data.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Kaggle retail/sales data
Target: Sales amount

## 2 · Project Overview

We predict **future sales** using retail/sales data. Sales forecasting is a core business problem — accurate predictions improve inventory management, staffing, and revenue planning. We treat this as a tabular regression task.

## 3 · Learning Objectives

1. Work with retail/sales datasets.
2. Handle date-based feature engineering.
3. Apply regression models to sales forecasting.
4. Use LazyPredict and FLAML for model selection.
5. Understand business impact of sales predictions.

## 4 · Problem Statement

Predict **future sales amounts** from historical sales records including product, store, and temporal features.

## 5 · Why This Project Matters

- **Sales forecasting** directly impacts supply chain and inventory decisions.
- Overprediction leads to excess stock; underprediction leads to stockouts.
- One of the most common ML use cases in retail and e-commerce.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | Kaggle: rohitsahoo/sales-forecasting |
| **Target** | Sales amount |
| **Features** | Ship Mode, Segment, Country, City, State, Category, Sub-Category, etc. |

## 7 · Dataset Source and License Notes

- **Source**: Kaggle Sales Forecasting dataset.
- **License**: Educational use.
- **Note**: Contains US retail sales data with geographic and product category features.

## 8 · Environment Setup

In [ ]:
import subprocess, sys
def _install_if_missing(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
for p, i in [('catboost',None),('lightgbm',None),('xgboost',None),('flaml',None),('lazypredict',None)]:
    _install_if_missing(p, i)
_install_if_missing('kagglehub')
print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)
warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
MAX_ROWS = 50000
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
import kagglehub, glob

try:
    path = kagglehub.dataset_download('rohitsahoo/sales-forecasting')
    print(f'Downloaded to: {path}')
except Exception as e:
    print(f'Download error: {e}')
    path = SAVE_DIR

csv_files = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
assert csv_files, f'No CSV found in {path}'
df = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0], encoding='latin1')
print(f'Loaded: {df.shape}')
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Columns: {list(df.columns)}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')

## 13 · Exploratory Data Analysis

In [ ]:
# Find the target column
target_candidates = [c for c in df.columns if any(kw in c.lower() for kw in ['sales', 'revenue', 'amount', 'profit'])]
if target_candidates:
    TARGET = target_candidates[0]
else:
    num_cols = df.select_dtypes(include='number').columns.tolist()
    TARGET = num_cols[-1] if num_cols else df.columns[-1]

print(f'Target: {TARGET}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].hist(bins=50, ax=axes[0], edgecolor='black')
axes[0].set_title(f'{TARGET} Distribution')
if 'Category' in df.columns:
    df.groupby('Category')[TARGET].mean().sort_values().plot.barh(ax=axes[1])
    axes[1].set_title(f'Avg {TARGET} by Category')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

## 16 · Preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Parse date columns
for col in df.columns:
    if 'date' in col.lower() or 'order' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col], errors='coerce')
            if df[col].dtype == 'datetime64[ns]':
                df[f'{col}_year'] = df[col].dt.year
                df[f'{col}_month'] = df[col].dt.month
                df[f'{col}_day'] = df[col].dt.day
                df = df.drop(columns=[col])
        except: pass

# Drop ID/name columns + high cardinality
for col in df.select_dtypes(include='object').columns:
    nuniq = df[col].nunique()
    if nuniq > 100 or nuniq < 2 or 'id' in col.lower() or 'name' in col.lower():
        df = df.drop(columns=[col])
    else:
        df[col] = df[col].fillna('Unknown')
        df[col] = LabelEncoder().fit_transform(df[col])

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(f'Preprocessed: {df.shape}')

## 17 · Feature Engineering

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Replace inf with NaN, then fill
X = X.replace([np.inf, -np.inf], np.nan)
for col in X.columns:
    X[col] = X[col].fillna(X[col].median() if X[col].median() == X[col].median() else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)
baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train.head(min(5000,len(X_train))), X_test.head(min(1000,len(X_test))),
                           y_train.head(min(5000,len(y_train))), y_test.head(min(1000,len(y_test))))
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(X_train, y_train, task='regression', time_budget=60, metric='rmse', seed=SEED, verbose=0)
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}, MAE: {flaml_mae:.2f}, R²: {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Boosting Models

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
models = {
    'CatBoost': CatBoostRegressor(iterations=300, learning_rate=0.1, depth=6, random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=SEED, verbosity=0)
}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)
ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")
top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Eval of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}: RMSE={m['RMSE']:.2f}, MAE={m['MAE']:.2f}, R²={m['R2']:.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models: best_model = models[best_name]
else: best_model = models.get('CatBoost', baseline)
y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5); axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual'); axes[0].set_title('Residuals vs Predicted')
axes[1].hist(residuals, bins=30, edgecolor='black'); axes[1].set_title('Residual Distribution')
axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].set_title('Actual vs Predicted')
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100); plt.show()

## 25 · Interpretation and Business Insight

- **Category** and **Sub-Category** are typically the strongest drivers.
- Geographic features (region, state) show distinct sales patterns.
- Discount and quantity interact to affect total sales.
- These models help retailers with demand planning and inventory allocation.

## 26 · Limitations

- Historical data doesn't capture market shifts or new products.
- No external factors (economy, competition, seasonality beyond dates).
- Sales are aggregated — individual transaction prediction would differ.

## 27 · How to Improve

1. Add holiday/event features.
2. Include economic indicators.
3. Add product-level time-series lags.
4. External data: weather, promotions, competitor pricing.

## 28 · Production Considerations

- Regular retraining as product mix changes.
- Integration with inventory management systems.
- Batch vs real-time prediction pipeline.
- Monitoring for concept drift in sales patterns.

## 29 · Common Mistakes

1. Including Order ID as a feature.
2. Not parsing date columns properly.
3. Ignoring the heavy right skew in sales.
4. Confusing sales prediction with time-series forecasting.

## 30 · Mini Challenge

1. Try log-transforming the target.
2. Add interaction features (Category × Region).
3. Build separate models per product category.
4. Add rolling average features if dates are available.

## 31 · Final Summary

- Sales prediction is a high-impact retail ML problem.
- Feature engineering from categorical and date features matters significantly.
- Gradient-boosting models handle the mixed feature types well.

In [ ]:
metrics = {}
for name in top3_names: metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')